# Imports

In [ ]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
import time
from matplotlib.backends.backend_pdf import PdfPages
from analyze.tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, set_seed, timing_decorator
from data_provider.data_loader import Dataset_Custom, Dataset_ETT_hour, Dataset_PEMS_PCA, Dataset_ETT_minute
from utils.polynomial import get_pca_base, Basis_Cache, pca_torch
import json

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# PCA complexity

In [ ]:
dst = 'Weather'
root_path = '/data/home/Licheng/workspace/TSF-CCA/dataset/weather'
data_path = 'weather.csv'
flag = 'train'

os.environ['CUDA_VISIBLE_DEVICES'] = '3'

def collect_label_seq(train_data, pl):
    label_seq = []
    for i in range(len(train_data)):
        _, label, _, _ = train_data[i]
        label = label[-pl:]
        label_seq.append(label)
    label_seq = np.array(label_seq)
    return label_seq


def test_time(pl, dp):
    set_seed(2023)
    train_data = Dataset_Custom(
        root_path=root_path,
        data_path=data_path,
        flag=flag,
        size=[96, 48, pl],
        features="M", target='OT', timeenc=1, freq='h',
        data_percentage=dp,
    )
    label_seq = collect_label_seq(train_data, pl)
    start = time.time()
    get_pca_base(label_seq, rank_ratio=1.0, pca_dim='T', reinit=1, speedup_sklearn=2)
    end = time.time()
    return end - start


def collect_data(trials, test_params):
    results = []
    for pl, dp in test_params:
        for _ in range(trials):
            t = test_time(pl, dp)
            results.append({'pred_len': pl, 'data_percentage': dp, 'time': t * 1000})  # Convert to milliseconds
    return pd.DataFrame(results)


def remove_extreme_samples(group):
    threshold = 4
    if len(group) <= threshold:
        return pd.DataFrame(columns=group.columns)
    return group.nlargest(len(group) - threshold, 'time').nsmallest(len(group) - threshold, 'time')


# Parameters setup
trials = 20

# Data collection
# test_params = list(product([96, 192, 336, 720], [0.2, 0.4, 0.6, 0.8, 1.0]))
# data = collect_data(trials, test_params)

# # Removing extreme samples and preparing final plots
# data_clean = data.groupby(['pred_len', 'data_percentage']).apply(remove_extreme_samples).reset_index(drop=True)
# save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
# data_clean.to_csv(os.path.join(save_root, f'weather_pca_time_rebb.csv'), index=False)
# data_clean



test_params = list(product([96, 192, 336, 720], [7200, 14400, 21600, 28800, 36000]))
data = collect_data(trials, test_params)

# Removing extreme samples and preparing final plots
data_clean = data.groupby(['pred_len', 'data_percentage']).apply(remove_extreme_samples).reset_index(drop=True)
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
data_clean.to_csv(os.path.join(save_root, f'weather_pca_time_rebb2.csv'), index=False)
data_clean

In [ ]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
# data_clean = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb.csv'))
data_clean = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb2.csv'))
data_clean['time'] /= 1000  # Convert to seconds
data_clean.head(10)

In [ ]:
df = data_clean.groupby(['pred_len', 'data_percentage']).head(5).reset_index(drop=True)
# df['samples'] = df['data_percentage'] * 36886
# df['samples'] = df['samples'].astype(int)
# df.drop(columns=['data_percentage'], inplace=True)
df.columns = ['pred_len', 'samples', 'time']

# 计算均值与标准差
agg = df.groupby(['pred_len', 'samples'])['time'].agg(['mean', 'std']).reset_index()

# 组合成 "mean±std" 形式
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")

# 透视表，行=pl，列=dp，值=result
pivot = agg.pivot(index='samples', columns='pred_len', values='result')

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
pivot.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_agg2.csv'), index=True)
pivot

: 

# load Fredformer data and plot

In [ ]:
import torch
import torch.nn as nn

dst = 'Weather'
root_path = '/data/home/Licheng/workspace/TSF-CCA/dataset/weather'
data_path = 'weather.csv'
flag = 'train'
size = [96, 48]
sl = 96
trys = 100
device = 'cuda:0'
batch_size = 2048
d = 21

os.environ['CUDA_VISIBLE_DEVICES'] = '3'


def collect_label_seq(train_data, pl):
    label_seq = []
    for i in range(len(train_data)):
        _, label, _, _ = train_data[i]
        label = label[-pl:]
        label_seq.append(label)
    label_seq = np.array(label_seq)
    return label_seq


kwargs_collection = {}
for pl in [96, 192, 336, 720]:
    set_seed(2023)
    train_data = Dataset_Custom(
        root_path=root_path,
        data_path=data_path,
        flag=flag,
        size=[96, 48, pl],
        features="M", target='OT', timeenc=1, freq='h',
        data_percentage=1.0,
    )
    label_seq = collect_label_seq(train_data, pl)
    pca_components, initializer, weights = get_pca_base(label_seq, rank_ratio=1.0, pca_dim='T', reinit=1, speedup_sklearn=2)
    pca_cache = Basis_Cache(pca_components, initializer, weights=weights, device=device)
    kwargs = {'pca_dim': 'T', 'pca_cache': pca_cache, 'use_weights': 0, "reinit": 1, "device": device}
    kwargs_collection[pl] = kwargs


criterion = nn.MSELoss()

def test_time(pl, mode):
    kwargs = kwargs_collection[pl]
    set_seed(2023)

    outputs = torch.randn(batch_size, pl, d, device=device, requires_grad=True).float()
    batch_y = torch.randn(batch_size, pl, d, device=device).float()
    start = time.time()
    if mode == "TransDF":
        loss = pca_torch(outputs, **kwargs) - pca_torch(batch_y, **kwargs)
        loss = loss.abs().mean()
    else:
        loss = criterion(outputs, batch_y)
    end = time.time()
    forward_time = end - start

    start = time.time()
    loss.backward()
    end = time.time()
    backward_time = end - start
    return forward_time, backward_time


def collect_data(trials, test_params):
    results = []
    for pl, mode in test_params:
        for _ in range(trials):
            t1, t2 = test_time(pl, mode)
            results.append({'pred_len': pl, 'mode': mode, 'forward_time': t1 * 1000, 'backward_time': t2 * 1000})  # Convert to milliseconds
    return pd.DataFrame(results)


def remove_extreme_samples(group):
    threshold = 20
    if len(group) <= threshold:
        return pd.DataFrame(columns=group.columns)
    return group.nlargest(len(group) - threshold, 'time').nsmallest(len(group) - threshold, 'time')


trials = 100

test_params = list(product([96, 192, 336, 720], ["TransDF", "DF"]))
data = collect_data(trials, test_params)

data_f = data[['pred_len', 'mode', 'forward_time']].copy()
data_f.columns = ['pred_len', 'mode', 'time']
data_b = data[['pred_len', 'mode', 'backward_time']].copy()
data_b.columns = ['pred_len', 'mode', 'time']

data_f = data_f.groupby(['pred_len', 'mode']).apply(remove_extreme_samples).reset_index(drop=True)
data_b = data_b.groupby(['pred_len', 'mode']).apply(remove_extreme_samples).reset_index(drop=True)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
data_f.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward.csv'), index=False)
data_b.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward.csv'), index=False)

In [ ]:
data_f = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward.csv'))
data_b = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward.csv'))

N = 60
df1 = data_f.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df1.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot1 = agg.pivot(index='mode', columns='pred_len', values='result')

df2 = data_b.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df2.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot2 = agg.pivot(index='mode', columns='pred_len', values='result')

pivot1['direction'] = 'forward'
pivot2['direction'] = 'backward'

pivot1 = pivot1.reset_index().set_index(['direction', 'mode'])
pivot2 = pivot2.reset_index().set_index(['direction', 'mode'])

pivot = pd.concat([pivot1, pivot2])
pivot

In [ ]:
import torch
import torch.nn as nn

dst = 'Weather'
root_path = '/data/home/Licheng/workspace/TSF-CCA/dataset/weather'
data_path = 'weather.csv'
flag = 'train'
size = [96, 48]
sl = 96
trys = 100
device = 'cuda:0'
batch_size = 2048
d = 21

os.environ['CUDA_VISIBLE_DEVICES'] = '3'


def collect_label_seq(train_data, pl):
    label_seq = []
    for i in range(len(train_data)):
        _, label, _, _ = train_data[i]
        label = label[-pl:]
        label_seq.append(label)
    label_seq = np.array(label_seq)
    return label_seq


kwargs_collection = {}
for pl in [96, 192, 336, 720]:
    set_seed(2023)
    train_data = Dataset_Custom(
        root_path=root_path,
        data_path=data_path,
        flag=flag,
        size=[96, 48, pl],
        features="M", target='OT', timeenc=1, freq='h',
        data_percentage=1.0,
    )
    label_seq = collect_label_seq(train_data, pl)
    pca_components, initializer, weights = get_pca_base(label_seq, rank_ratio=1.0, pca_dim='T', reinit=1, speedup_sklearn=2)
    pca_cache = Basis_Cache(pca_components, initializer, weights=weights, device=device)
    kwargs = {'pca_dim': 'T', 'pca_cache': pca_cache, 'use_weights': 0, "reinit": 1, "device": device}
    kwargs_collection[pl] = kwargs


criterion = nn.MSELoss()

def test_time(pl, mode):
    kwargs = kwargs_collection[pl]
    set_seed(2023)

    outputs = torch.randn(batch_size, pl, d, device=device, requires_grad=True).float()
    batch_y = torch.randn(batch_size, pl, d, device=device).float()
    start = time.time()
    if mode == "TransDF":
        loss = pca_torch(outputs, **kwargs) - pca_torch(batch_y, **kwargs)
        loss = loss.abs().mean()
    else:
        loss = criterion(outputs, batch_y)
    end = time.time()
    forward_time = end - start

    start = time.time()
    loss.backward()
    end = time.time()
    backward_time = end - start
    return forward_time, backward_time


def collect_data(trials, test_params):
    results = []
    for pl, mode in test_params:
        for _ in range(trials):
            t1, t2 = test_time(pl, mode)
            results.append({'pred_len': pl, 'mode': mode, 'forward_time': t1 * 1000, 'backward_time': t2 * 1000})  # Convert to milliseconds
    return pd.DataFrame(results)


def remove_extreme_samples(group):
    threshold = 20
    if len(group) <= threshold:
        return pd.DataFrame(columns=group.columns)
    return group.nlargest(len(group) - threshold, 'time').nsmallest(len(group) - threshold, 'time')


trials = 400

test_params = list(product([720], ["TransDF", "DF"]))
data = collect_data(trials, test_params)

data_f = data[['pred_len', 'mode', 'forward_time']].copy()
data_f.columns = ['pred_len', 'mode', 'time']
data_b = data[['pred_len', 'mode', 'backward_time']].copy()
data_b.columns = ['pred_len', 'mode', 'time']

data_f = data_f.groupby(['pred_len', 'mode']).apply(remove_extreme_samples).reset_index(drop=True)
data_b = data_b.groupby(['pred_len', 'mode']).apply(remove_extreme_samples).reset_index(drop=True)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
data_f.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_720.csv'), index=False)
data_b.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward_720.csv'), index=False)

In [ ]:
data_f = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_720.csv'))
data_b = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward_720.csv'))

N = 200
df1 = data_f.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df1.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot1 = agg.pivot(index='mode', columns='pred_len', values='result')

df2 = data_b.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df2.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot2 = agg.pivot(index='mode', columns='pred_len', values='result')

pivot1['direction'] = 'forward'
pivot2['direction'] = 'backward'

pivot1 = pivot1.reset_index().set_index(['direction', 'mode'])
pivot2 = pivot2.reset_index().set_index(['direction', 'mode'])

pivot_720 = pd.concat([pivot1, pivot2])


data_f = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward.csv'))
data_b = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward.csv'))

data_f = data_f[data_f['pred_len'] != 720]
data_b = data_b[data_b['pred_len'] != 720]

N = 60
df1 = data_f.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df1.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot1 = agg.pivot(index='mode', columns='pred_len', values='result')

df2 = data_b.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df2.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot2 = agg.pivot(index='mode', columns='pred_len', values='result')

pivot1['direction'] = 'forward'
pivot2['direction'] = 'backward'

pivot1 = pivot1.reset_index().set_index(['direction', 'mode'])
pivot2 = pivot2.reset_index().set_index(['direction', 'mode'])

pivot_other = pd.concat([pivot1, pivot2])

pivot = pd.concat([pivot_other, pivot_720], axis=1)
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
pivot.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_backward.csv'), index=True)
pivot

# add fredf

In [ ]:
import torch
import torch.nn as nn

dst = 'Weather'
root_path = '/data/home/Licheng/workspace/TSF-CCA/dataset/weather'
data_path = 'weather.csv'
flag = 'train'
size = [96, 48]
sl = 96
trys = 100
device = 'cuda:0'
batch_size = 2048
d = 21

os.environ['CUDA_VISIBLE_DEVICES'] = '3'


criterion = nn.MSELoss()

def test_time(pl, mode):
    set_seed(2023)

    outputs = torch.randn(batch_size, pl, d, device=device, requires_grad=True).float()
    batch_y = torch.randn(batch_size, pl, d, device=device).float()
    start = time.time()
    if mode == "TransDF":
        loss = pca_torch(outputs, **kwargs) - pca_torch(batch_y, **kwargs)
        loss = loss.abs().mean()
    elif mode == "DF":
        loss = criterion(outputs, batch_y)
    elif mode == "FreDF":
        loss = torch.fft.rfft(outputs, dim=1) - torch.fft.rfft(batch_y, dim=1)  # shape: [B, P//2+1, D]
        loss = loss.abs().mean()  # Assuming you want to take the absolute value and mean
    end = time.time()
    forward_time = end - start

    start = time.time()
    loss.backward()
    end = time.time()
    backward_time = end - start
    return forward_time, backward_time


def collect_data(trials, test_params):
    results = []
    for pl, mode in test_params:
        for _ in range(trials):
            t1, t2 = test_time(pl, mode)
            results.append({'pred_len': pl, 'mode': mode, 'forward_time': t1 * 1000, 'backward_time': t2 * 1000})  # Convert to milliseconds
    return pd.DataFrame(results)


def remove_extreme_samples(group):
    threshold = 20
    if len(group) <= threshold:
        return pd.DataFrame(columns=group.columns)
    return group.nlargest(len(group) - threshold, 'time').nsmallest(len(group) - threshold, 'time')


trials = 400

test_params = list(product([96, 192, 336, 720], ["FreDF"]))
data = collect_data(trials, test_params)

data_f = data[['pred_len', 'mode', 'forward_time']].copy()
data_f.columns = ['pred_len', 'mode', 'time']
data_b = data[['pred_len', 'mode', 'backward_time']].copy()
data_b.columns = ['pred_len', 'mode', 'time']

data_f = data_f.groupby(['pred_len', 'mode']).apply(remove_extreme_samples).reset_index(drop=True)
data_b = data_b.groupby(['pred_len', 'mode']).apply(remove_extreme_samples).reset_index(drop=True)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
data_f.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_fredf.csv'), index=False)
data_b.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward_fredf.csv'), index=False)

In [ ]:
data_f = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_720.csv'))
data_b = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward_720.csv'))

N = 200
df1 = data_f.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df1.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot1 = agg.pivot(index='mode', columns='pred_len', values='result')

df2 = data_b.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df2.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot2 = agg.pivot(index='mode', columns='pred_len', values='result')

pivot1['direction'] = 'forward'
pivot2['direction'] = 'backward'

pivot1 = pivot1.reset_index().set_index(['direction', 'mode'])
pivot2 = pivot2.reset_index().set_index(['direction', 'mode'])

pivot_720 = pd.concat([pivot1, pivot2])


data_f = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward.csv'))
data_b = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward.csv'))

data_f = data_f[data_f['pred_len'] != 720]
data_b = data_b[data_b['pred_len'] != 720]

N = 60
df1 = data_f.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df1.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot1 = agg.pivot(index='mode', columns='pred_len', values='result')

df2 = data_b.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df2.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot2 = agg.pivot(index='mode', columns='pred_len', values='result')

pivot1['direction'] = 'forward'
pivot2['direction'] = 'backward'

pivot1 = pivot1.reset_index().set_index(['direction', 'mode'])
pivot2 = pivot2.reset_index().set_index(['direction', 'mode'])

pivot_other = pd.concat([pivot1, pivot2])
pivot = pd.concat([pivot_other, pivot_720], axis=1)


data_f = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_fredf.csv'))
data_b = pd.read_csv(os.path.join(save_root, f'weather_pca_time_rebb_backward_fredf.csv'))

N = 200
df1 = data_f.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df1.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot1 = agg.pivot(index='mode', columns='pred_len', values='result')

df2 = data_b.groupby(['pred_len', 'mode']).head(N).reset_index(drop=True)
agg = df2.groupby(['pred_len', 'mode'])['time'].agg(['mean', 'std']).reset_index()
agg['result'] = agg['mean'].map(lambda x: f"{x:.3f}") + '±' + agg['std'].map(lambda x: f"{x:.3f}")
pivot2 = agg.pivot(index='mode', columns='pred_len', values='result')

pivot1['direction'] = 'forward'
pivot2['direction'] = 'backward'

pivot1 = pivot1.reset_index().set_index(['direction', 'mode'])
pivot2 = pivot2.reset_index().set_index(['direction', 'mode'])

pivot_fredf = pd.concat([pivot1, pivot2])

pivot = pd.concat([pivot, pivot_fredf], axis=0)
pivot.index = pd.MultiIndex.from_frame(
    pivot.index.to_frame().assign(
        mode=pd.Categorical(
            pivot.index.get_level_values('mode'),
            categories=['DF', 'FreDF', 'TransDF'],
            ordered=True
        )
    )
)
pivot = pivot.sort_index(level=['direction', 'mode'])
pivot

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
pivot.to_csv(os.path.join(save_root, f'weather_pca_time_rebb_forward_backward_three.csv'), index=True)
pivot